In [2]:
import json
import pandas as pd
import numpy as np

# -------------------------------
# LOAD JSON DATA
# -------------------------------
with open("/content/noisy_amazon_data.json", "r", encoding="utf-8") as f:
    noisy_data = json.load(f)

print("Total records (with noise):", len(noisy_data))

# Normalize JSON
df = pd.json_normalize(noisy_data)

df.head()

# -------------------------------
# BASIC INFO
# -------------------------------
df.info()
df.isna().sum()

# -------------------------------
# TEXT CLEANING FUNCTION
# -------------------------------
def clean_text(series):
    return (
        series.astype(str)
        .str.lower()
        .str.strip()
        .str.replace(r"[^a-z0-9\s]", "", regex=True)
    )

df["title"] = clean_text(df["title"])
df["brand"] = clean_text(df["brand"])

# -------------------------------
# HANDLE MISSING VALUES
# -------------------------------
df["description"] = df["description"].replace(["", "none", "nan"], np.nan)

df["brand"] = df["brand"].fillna("unknown")

# -------------------------------
# CLEAN 'stars' COLUMN
# -------------------------------
df["stars"] = pd.to_numeric(df["stars"], errors="coerce")

df.loc[(df["stars"] < 0) | (df["stars"] > 5), "stars"] = np.nan

df["stars"] = df["stars"].fillna(df["stars"].mean())

# -------------------------------
# CLEAN 'reviewsCount'
# -------------------------------
df["reviewsCount"] = pd.to_numeric(df["reviewsCount"], errors="coerce")
df["reviewsCount"] = df["reviewsCount"].fillna(0)

# -------------------------------
# CLEAN PRICE VALUE
# -------------------------------
df["price.value"] = (
    df["price.value"]
    .astype(str)
    .str.replace(r"[^0-9.]", "", regex=True)
)

df["price.value"] = pd.to_numeric(df["price.value"], errors="coerce")

df.loc[df["price.value"] <= 0, "price.value"] = np.nan

df["price.value"] = df["price.value"].fillna(df["price.value"].median())

# -------------------------------
# CLEAN PRICE CURRENCY
# -------------------------------
df["price.currency"] = df["price.currency"].fillna("$")

# -------------------------------
# REMOVE DUPLICATES
# -------------------------------
before = len(df)

df = df.drop_duplicates(subset=["asin"])

after = len(df)

print("Duplicates removed:", before - after)

# -------------------------------
# FINAL CHECK
# -------------------------------
df.info()
df.isna().sum()

# -------------------------------
# SAVE CLEAN DATA
# -------------------------------
df.to_json("/content/clean_amazon_data.json", orient="records", indent=2)

Total records (with noise): 16
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 16 entries, 0 to 15
Data columns (total 8 columns):
 #   Column          Non-Null Count  Dtype 
---  ------          --------------  ----- 
 0   asin            16 non-null     object
 1   title           16 non-null     object
 2   brand           15 non-null     object
 3   stars           16 non-null     object
 4   reviewsCount    15 non-null     object
 5   description     16 non-null     object
 6   price.value     16 non-null     object
 7   price.currency  15 non-null     object
dtypes: object(8)
memory usage: 1.1+ KB
Duplicates removed: 1
<class 'pandas.core.frame.DataFrame'>
Index: 15 entries, 0 to 14
Data columns (total 8 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   asin            15 non-null     object 
 1   title           15 non-null     object 
 2   brand           15 non-null     object 
 3   stars           15 non-null     float6